# ಪಾಠ 04 - ಸಾಧನ ಬಳಕೆ ವಿನ್ಯಾಸ ಮಾದರಿ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework (Python) ಬಳಸಿ AI ಏಜೆಂಟ್‌ಗಳಿಗೆ **ಸಾಧನ ಬಳಕೆ** ವಿನ್ಯಾಸ ಮಾದರಿಯನ್ನು ಕಲಿಯುತ್ತೀರಿ. ನಾವು ಕವರ್ ಮಾಡುತ್ತೇವೆ:

- `@tool` ಅಲಂಕಾರ ಮತ್ತು ಟೈಪ್ ಮಾಡಿದ ಪ್ಯಾರಾಮೀಟರ್‌ಗಳೊಂದಿಗೆ ಫಂಕ್ಷನ್ ಸಾಧನಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು
- ಪ್ರತಿ ಸಾಧನ ಏನೆಂದು ಮಾದರಿ ತಿಳಿಯುವಂತೆ ಸಾಧನ schemas ಒದಗಿಸುವುದು
- `approval_mode` ಮೂಲಕ ಸಾಧನ ಕಾರ್ಯಾಚರಣೆಯನ್ನು ನಿಯಂತ್ರಿಸುವುದು
- Pydantic ಮಾದರಿಗಳ ಮೂಲಕ ಮತ್ತು `response_format` ಮೂಲಕ **ಸಂರಚಿತ ಔಟ್‌ಪುಟ್** ನೀಡುವುದು

ಈ ಸಂದರ್ಭವಿರುವುದು **ಪ್ರಯಾಣ ಬುಕಿಂಗ್ ಏಜೆಂಟ್** ಆಗಿದ್ದು, ಗಮ್ಯಸ್ಥಾನಗಳನ್ನು ಹುಡುಕುವುದು, ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುವುದು, ಮತ್ತು ವಿಮಾನ ಮಾಹಿತಿಯನ್ನು ಪಡೆಯುವುದು.


## ಸ್ಥಾಪನೆ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ಅಲಂಕಾರದೊಂದಿಗೆ ಉಪಕರಣಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು

`@tool` ಅಲಂಕಾರವು ಸರಳ Python ಕಾರ್ಯವನ್ನು ಏজಂಟ್ ಕರೆಮಾಡಬಹುದಾದ ಉಪಕರಣವಾಗಿಸುತ್ತದೆ.  
ಪ್ರಮುಖ ಅಂಶಗಳು:

- **ಡಾಕ್‌ಸ್ಟ್ರಿಂಗ್** ಮಾದರಿ ನೋಡಿಕೊಳ್ಳುವ ಉಪಕರಣ ವಿವರಣೆಯಾಗುತ್ತದೆ.  
- **ಪ್ರಕಾರ ಟಿಪ್ಪಣಿಗಳು** (`Annotated` ವಿವರಣೆಗಳೊಂದಿಗೆ ಸೇರಿ) ಉಪಕರಣದ ಯೋಜನೆಯನ್ನು ನಿರ್ಧರಿಸುತ್ತವೆ.  
- `approval_mode` ಬಳಕೆದಾರನು ಪ್ರತಿ ಕರೆ ಕಾರ್ಯಗತಗೊಳ್ಳುವುದಕ್ಕೂ ಮುನ್ನ ಅನುಮೋದನೆ ನೀಡಬೇಕೇ ಎಂಬುದನ್ನು ನಿಯಂತ್ರಿಸುತ್ತದೆ.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ಬಹು ಸಾಧನಗಳೊಂದಿಗೆ ಏಜೆಂಟ್ ರಚನೆ

ಮಾಡೆಲ್ ಬಳಕೆದಾರರ ಪ್ರಶ್ನೆಗೆ ಉತ್ತರಿಸಲು ಬೇಕಾದ ಯಾವುದೇ ಸಾಧನಗಳನ್ನು ಕರೆಿಸಲು, ಎಲ್ಲಾ ಮೂವರು ಸಾಧನಗಳನ್ನು ಕ್ಲೈಯಂಟ್‌ಗೆ ಪಾಸ್ ಮಾಡಿ.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ಸಾಧನಗಳೊಂದಿಗೆ ಸಂರಚಿತ ಔಟ್‌ಪುಟು

`response_format` ಅನ್ನು ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಯಾಗಿರಿಸುವ ಮೂಲಕ, ಏಜೆಂಟ್ ಸ್ವತಃ ಸ್ವೇಚ್ಛಾಶೀಲ ಪಠ್ಯದ ಬದಲು ಉತ್ತಮವಾಗಿ ಪ್ರಕಾರಬದ್ಧ JSON ವಸ್ತುವನ್ನು ಹಿಂತಿರುಗಿಸಲು ಬಾಧ್ಯನಾಗುತ್ತದೆ. ಇದು ನಂತರದ ಕೋಡ್ ಫಲಿತಾಂಶವನ್ನು ಪ್ರೋಗ್ರಾಮ್ಯಾಟ್‌ಕಾಗಿ ಬಳಸಿಕೊಳ್ಳಬೇಕಾದಾಗ ಉಪಯುಕ್ತವಾಗಿದೆ.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ಟೂಲ್ ಅನುಮೋದನೆ ಮಾದರಿಗಳು

`@tool` ನಲ್ಲಿ ಇರುವ `approval_mode` ಪರಿಮಿತಿ ಟೂಲ್ ಕರೆಯುವಿಕೆಗೆ ಅನುಮೋದನೆ ಅಗತ್ಯವಿದ್ದರೆ ಅದನ್ನು ನಿಯಂತ್ರಿಸುತ್ತದೆ:

| ಮೋಡ್ | ಪ್ರవర್ತನೆ |
|---|---|
| `"never_require"` | ಟೂಲ್ ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಓಡುತ್ತದೆ — ಬಳಕೆದಾರರ ದೃಢೀಕರಣ ಅಗತ್ಯವಿಲ್ಲ. |
| `"always_require"` | ಪ್ರತಿಯೊಂದು ಕರೆ ಬಳಕೆದಾರರಿಂದ ಅನುಮೋದನೆಗೆ ಒಳಪಟ್ಟ ನಂತರವೇ ಕಾರ್ಯಗತಗೊಳಿಸಲಾಗುತ್ತದೆ. |

ಬಾಹ್ಯ ಪರಿಣಾಮಗಳಿರುವ ಟೂಲ್‌ಗಳಿಗೆ (ಉದಾಹರಣೆಗೆ ವಿಮಾನ ಬುಕ್ಕಿಂಗ್, ಕ್ರೆಡಿಟ್ ಕಾರ್ಡ್‌ನಿಂದ ಪಾವತಿ) `"always_require"` ಬಳಸಿ, ಹಾಗಾದರೆ ಮಾನವ ನಿರಂತರ ನಿಯಂತ್ರಣದಲ್ಲಿ ಇರಬಹುದು.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಕೆಲವನ್ನು ಕಲಿತಿರಿ:

1. **ಟೂಲ್‌ಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವುದು** `@tool` ಡೆಕೊರೇಟರ್ ಬಳಸಿ ಟೈಪ್ಡ್ ಪ್ಯಾರಾಮೀಟರ್‌ಗಳು ಮತ್ತು ಟೂಲ್ ನಿಯಮವನ್ನು ಹೊಂದಿರುವ ಡಾಕ್ಸ್ಟ್ರಿಂಗ್‌ಗಳೊಂದಿಗೆ.
2. **ಹೆಚ್ಚು ಟೂಲ್‌ಗಳನ್ನು ರಚಿಸುವುದು** ಆಗಿದ್ದು ಏಜೆಂಟ್ ಅವುಗಳನ್ನು ಕ್ರಮಬದ್ಧವಾಗಿ ಕರೆ ಮಾಡಿ ಸಂಕೀರ್ಣ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರಿಸಬಹುದು.
3. **ರಚನೆಗೊಳ್ಳುವ ನಿರ್ಗಮನವನ್ನು ಹಿಂದಿರುಗಿಸುವುದು** Pydantic ಮಾದರಿಯನ್ನು `response_format` ಆಗಿ ಪಾಸಾಗಿ.
4. **ಟೂಲ್ ಅನುಮೋದನೆಯನ್ನು ನಿಯಂತ್ರಿಸುವುದು** `approval_mode` ಬಳಸಿಕೊಂಡು ಸಂವೇದನಾಶೀಲ ಕಾರ್ಯಕ್ಕಾಗಿ ಮಾನವನನ್ನು ಲೂಪ್‌ನಲ್ಲಿ ಇರಿಸಲು.

ಈ ಮಾದರಿಗಳು ವಿಶ್ವಾಸಾರ್ಹ, ಉತ್ಪಾದನೆಗೆ ಸಿದ್ಧವಾದ ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುವ ಮೂಲಾಧಾರವನ್ನು ರೂಪಿಸುತ್ತವೆ, ಅವು ಬಾಹ್ಯ ವ್ಯವಸ್ಥೆಗಳೊಂದಿಗೆ ಸುರಕ್ಷಿತವಾಗಿ ಸಂವಹನ ಮಾಡಬಹುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ತಿರಸ್ಕರಣಾ ಹೇಳಿಕೆ**:  
ಈ ದಾಖಲೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಎಂಬ AI ಅನುವಾದ ಸೇವೆಯನ್ನು ಬಳಸಿಕೊಂಡು ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯ ಕಡೆಗೆ ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಯಂತ್ರಮಾನವನ ಅನುವಾದದಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸಂಬದ್ಧತೆಗಳಿರಬಹುದೆಂಬ ವಿಷಯವನ್ನು ಗಮನದಲ್ಲಿ ಇಡಿ. ಮೂಲ ಭಾಷೆಯ ಮೂಲ ದಾಖಲೆ ಅಧಿಕಾರವಂತ ಮೂಲವಾಗಿರುತ್ತದೆ. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗಿದೆ. ಈ ಅನುವಾದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ಗಲತಿಯುಗಳಿಗಾಗಿ ನಾವು ಜವಾಬ್ದಾರಿಯಾಗಿರುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
